# Day3_22 · 기상청 API에서 로컬 웹앱까지

기상청 초단기예보 응답을 한 단계씩 확인하고, pandas 표와 그래프를 거쳐 로컬 웹 화면으로 연결합니다.

## 이 노트북에서 확인할 내용

- JSON 응답에서 예보 항목을 찾아 pandas 표로 바꿀 수 있습니다.
- 위치와 발표시각을 확인해 최신 초단기예보를 요청할 수 있습니다.
- Notebook의 결과표를 로컬 Streamlit 웹앱에서 표시하고 CSV로 저장할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-22-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-22-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-22-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
    current_folder / 'day3_beginner' / 'data',
    current_folder / 'Day3_beginner_final' / 'data',
]

In [ ]:
data_candidates

### D3B-22-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-22-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-22-06 · JSON 도구 불러오기

In [ ]:
import json

### D3B-22-07 · 운영체제 환경변수 도구 불러오기

In [ ]:
import os

### D3B-22-08 · Python 경로 도구 불러오기

In [ ]:
import sys

### D3B-22-09 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt

### D3B-22-10 · seaborn 불러오기

In [ ]:
import seaborn as sns

## 먼저 저장된 응답을 살펴보기

기상청에 요청하면 표가 아니라 여러 겹의 JSON 응답이 돌아옵니다. 먼저 같은 모양의 샘플 응답을 열어 어느 위치에 예보 항목이 들어 있는지 확인합니다.

### D3B-22-11 · 샘플 응답 파일 위치 정하기

In [ ]:
sample_path = DATA_DIR / 'kma_ultra_short_forecast_sample.json'

In [ ]:
sample_path

### D3B-22-12 · 샘플 응답 파일 확인

In [ ]:
sample_path.exists()

### D3B-22-13 · 샘플 JSON 읽기

In [ ]:
sample_payload = json.loads(sample_path.read_text(encoding='utf-8'))

In [ ]:
sample_payload.keys()

### D3B-22-14 · response 부분 선택

In [ ]:
response_part = sample_payload['response']

In [ ]:
response_part.keys()

### D3B-22-15 · 응답 상태 선택

In [ ]:
response_header = response_part['header']

In [ ]:
response_header

### D3B-22-16 · 응답 본문 선택

In [ ]:
response_body = response_part['body']

In [ ]:
response_body.keys()

### D3B-22-17 · 예보 항목 묶음 선택

In [ ]:
item_container = response_body['items']

In [ ]:
item_container.keys()

### D3B-22-18 · 예보 항목 목록 선택

In [ ]:
raw_items = item_container['item']

In [ ]:
raw_items[:2]

### D3B-22-19 · 예보 항목 수 확인

In [ ]:
len(raw_items)

### D3B-22-20 · 첫 항목의 이름 확인

In [ ]:
raw_items[0].keys()

JSON 안에서 `response → body → items → item` 순서로 이동해 예보 목록을 찾았습니다. 이제 이 목록을 pandas 표로 바꿉니다.

### D3B-22-21 · 예보 목록을 표로 바꾸기

In [ ]:
raw_forecast = pd.DataFrame(raw_items)

In [ ]:
raw_forecast

### D3B-22-22 · 예보 표의 크기 확인

In [ ]:
raw_forecast.shape

### D3B-22-23 · 예보 표의 열 이름 확인

In [ ]:
raw_forecast.columns

### D3B-22-24 · 항목 코드 목록 만들기

In [ ]:
category_names = raw_forecast['category'].unique()

In [ ]:
category_names

### D3B-22-25 · 항목 코드 설명표 만들기

In [ ]:
category_guide = pd.Series({
    'T1H': '기온(°C)',
    'REH': '습도(%)',
    'RN1': '1시간 강수량',
    'SKY': '하늘상태',
    'PTY': '강수형태',
    'WSD': '풍속(m/s)',
}, name='뜻')

In [ ]:
category_guide

### D3B-22-26 · 예보 날짜와 시각 붙이기

In [ ]:
forecast_at_text = raw_forecast['fcstDate'].astype(str) + raw_forecast['fcstTime'].astype(str).str.zfill(4)

In [ ]:
forecast_at_text.head()

### D3B-22-27 · 붙인 글자를 날짜 형식으로 바꾸기

In [ ]:
raw_forecast['forecast_at'] = pd.to_datetime(forecast_at_text, format='%Y%m%d%H%M')

In [ ]:
raw_forecast[['fcstDate', 'fcstTime', 'forecast_at']].head()

### D3B-22-28 · 확인할 열만 고르기

In [ ]:
long_forecast = raw_forecast[['forecast_at', 'category', 'fcstValue', 'nx', 'ny']]

In [ ]:
long_forecast.head(12)

## 한 시각을 한 행으로 바꾸기

현재 표에서는 같은 시각이 기온·습도·강수·바람만큼 여러 행에 반복됩니다. 그래프와 웹 화면에서는 한 시각을 한 행으로 보는 편이 편리합니다.

### D3B-22-29 · 시각은 행, 항목은 열로 바꾸기

In [ ]:
wide_forecast = long_forecast.pivot(index='forecast_at', columns='category', values='fcstValue').reset_index()

In [ ]:
wide_forecast

### D3B-22-30 · 바뀐 표의 크기 확인

In [ ]:
wide_forecast.shape

### D3B-22-31 · 바뀐 표의 열 이름 확인

In [ ]:
wide_forecast.columns

### D3B-22-32 · 기온을 숫자로 바꾸기

In [ ]:
wide_forecast['T1H'] = pd.to_numeric(wide_forecast['T1H'], errors='coerce')

In [ ]:
wide_forecast['T1H']

### D3B-22-33 · 습도를 숫자로 바꾸기

In [ ]:
wide_forecast['REH'] = pd.to_numeric(wide_forecast['REH'], errors='coerce')

In [ ]:
wide_forecast['REH']

### D3B-22-34 · 풍속을 숫자로 바꾸기

In [ ]:
wide_forecast['WSD'] = pd.to_numeric(wide_forecast['WSD'], errors='coerce')

In [ ]:
wide_forecast['WSD']

### D3B-22-35 · 하늘상태 이름표 만들기

In [ ]:
sky_names = {'1': '맑음', '3': '구름 많음', '4': '흐림'}

In [ ]:
sky_names

### D3B-22-36 · 하늘상태를 이름으로 바꾸기

In [ ]:
wide_forecast['sky_label'] = wide_forecast['SKY'].astype(str).map(sky_names)

In [ ]:
wide_forecast[['SKY', 'sky_label']]

### D3B-22-37 · 강수형태 이름표 만들기

In [ ]:
rain_type_names = {'0': '없음', '1': '비', '2': '비/눈', '3': '눈', '4': '소나기', '5': '빗방울', '6': '빗방울/눈날림', '7': '눈날림'}

In [ ]:
rain_type_names

### D3B-22-38 · 강수형태를 이름으로 바꾸기

In [ ]:
wide_forecast['rain_type_label'] = wide_forecast['PTY'].astype(str).map(rain_type_names)

In [ ]:
wide_forecast[['PTY', 'rain_type_label']]

### D3B-22-39 · 화면에 사용할 표 만들기

In [ ]:
display_forecast = wide_forecast[['forecast_at', 'T1H', 'REH', 'WSD', 'RN1', 'sky_label', 'rain_type_label']].rename(columns={
    'forecast_at': '예보 시각',
    'T1H': '기온(°C)',
    'REH': '습도(%)',
    'WSD': '풍속(m/s)',
    'RN1': '1시간 강수량',
    'sky_label': '하늘',
    'rain_type_label': '강수형태',
})

In [ ]:
display_forecast

### D3B-22-40 · 수치 열 요약 확인

In [ ]:
display_forecast[['기온(°C)', '습도(%)', '풍속(m/s)']].describe().round(1)

## 표를 그래프로 확인하기

각 그래프는 예보 시각에 따라 값이 어떻게 달라지는지 보여 줍니다.

### D3B-22-41 · 시간별 기온 그리기

In [ ]:
sns.lineplot(data=display_forecast, x='예보 시각', y='기온(°C)', marker='o')
plt.xticks(rotation=30)
plt.title('Temperature forecast')
plt.show()

### D3B-22-42 · 시간별 습도 그리기

In [ ]:
sns.lineplot(data=display_forecast, x='예보 시각', y='습도(%)', marker='o', color='teal')
plt.xticks(rotation=30)
plt.title('Humidity forecast')
plt.show()

### D3B-22-43 · 시간별 풍속 그리기

In [ ]:
sns.barplot(data=display_forecast, x='예보 시각', y='풍속(m/s)', color='steelblue')
plt.xticks(rotation=30)
plt.title('Wind speed forecast')
plt.show()

### D3B-22-44 · 비가 예상된 시각 찾기

In [ ]:
rain_forecast = display_forecast[display_forecast['강수형태'] != '없음']

In [ ]:
rain_forecast

### D3B-22-45 · 가장 높은 기온의 행 찾기

In [ ]:
warmest_forecast = display_forecast.nlargest(1, '기온(°C)')

In [ ]:
warmest_forecast

## 실제 API 요청으로 연결하기

지금까지는 저장된 응답을 사용했습니다. 이제 위치의 격자번호와 최근 발표시각을 요청에 넣고, 준비된 키가 있으면 같은 구조의 최신 응답을 가져옵니다. 키 값 자체는 화면에 표시하지 않습니다.

### D3B-22-46 · 웹앱 폴더 위치 정하기

In [ ]:
APP_DIR = DATA_DIR.parent / 'apps' / 'kma_weather_local'

In [ ]:
APP_DIR

### D3B-22-47 · 웹앱 폴더 확인

In [ ]:
APP_DIR.exists()

### D3B-22-48 · 웹앱 파일 목록 만들기

In [ ]:
app_files = sorted(path.name for path in APP_DIR.iterdir() if not path.name.startswith('.'))

In [ ]:
app_files

### D3B-22-49 · 웹앱 도구 폴더를 Python에 연결

In [ ]:
sys.path.insert(0, str(APP_DIR)) if str(APP_DIR) not in sys.path else None

### D3B-22-50 · 기상청 요청 도구 불러오기

In [ ]:
from kma_client import latest_base_candidates, load_weather_or_sample, long_to_wide_frame, payload_to_long_frame, read_auth_key

### D3B-22-51 · 개인 키 파일 위치 정하기

In [ ]:
env_path = APP_DIR / '.env'

In [ ]:
env_path

### D3B-22-52 · 저장된 키 읽기

In [ ]:
api_key = read_auth_key(env_path)

In [ ]:
pd.Series({'KMA_AUTH_KEY 저장됨': bool(api_key)})

### D3B-22-53 · 위치별 격자표 만들기

In [ ]:
locations = pd.DataFrame([
    {'location': '울산대학교', 'nx': 101, 'ny': 84},
    {'location': '울산광역시 중심', 'nx': 102, 'ny': 84},
    {'location': '서울광역시 중심', 'nx': 60, 'ny': 127},
    {'location': '부산광역시 중심', 'nx': 98, 'ny': 76},
])

In [ ]:
locations

### D3B-22-54 · 확인할 위치 정하기

In [ ]:
# 다른 위치를 확인하려면 이름을 표에 있는 값으로 바꾸세요.
selected_location_name = '울산대학교'

In [ ]:
selected_location_name

### D3B-22-55 · 선택한 위치의 한 행 찾기

In [ ]:
selected_location = locations.set_index('location').loc[selected_location_name]

In [ ]:
selected_location

### D3B-22-56 · 최근 발표시각 후보 만들기

In [ ]:
base_candidates = pd.DataFrame(latest_base_candidates())

In [ ]:
base_candidates

### D3B-22-57 · 요청에 들어갈 공개 항목 표 만들기

In [ ]:
request_preview = pd.Series({
    'dataType': 'JSON',
    'numOfRows': 1000,
    'base_date': base_candidates.loc[0, 'base_date'],
    'base_time': base_candidates.loc[0, 'base_time'],
    'nx': int(selected_location['nx']),
    'ny': int(selected_location['ny']),
    'authKey': '화면에 표시하지 않음',
})

In [ ]:
request_preview

### D3B-22-58 · 최신 응답 가져오기

In [ ]:
weather_result = load_weather_or_sample(
    auth_key=api_key,
    nx=int(selected_location['nx']),
    ny=int(selected_location['ny']),
    sample_path=sample_path,
)

In [ ]:
weather_result.summary()

### D3B-22-59 · 가져온 응답을 긴 표로 바꾸기

In [ ]:
api_long_forecast = payload_to_long_frame(weather_result.payload)

In [ ]:
api_long_forecast.head(12)

### D3B-22-60 · 가져온 항목 수 확인

In [ ]:
api_long_forecast.shape

### D3B-22-61 · 가져온 응답을 시간별 표로 바꾸기

In [ ]:
api_forecast = long_to_wide_frame(api_long_forecast)

In [ ]:
api_forecast

### D3B-22-62 · 가져온 기온 그리기

In [ ]:
sns.lineplot(data=api_forecast, x='예보 시각', y='기온(°C)', marker='o')
plt.xticks(rotation=30)
plt.title(f'{selected_location_name} temperature forecast')
plt.show()

### D3B-22-63 · 가져온 예보의 가장 이른 시각 선택

In [ ]:
first_forecast = api_forecast.sort_values('예보 시각').head(1)

In [ ]:
first_forecast

### D3B-22-64 · 가져온 예보의 가장 높은 기온 선택

In [ ]:
highest_temperature = api_forecast.nlargest(1, '기온(°C)')

In [ ]:
highest_temperature

## 결과를 로컬 파일로 저장하기

### D3B-22-65 · 결과 폴더 위치 정하기

In [ ]:
output_dir = DATA_DIR.parent / 'output'

In [ ]:
output_dir

### D3B-22-66 · 결과 폴더 만들기

In [ ]:
output_dir.mkdir(exist_ok=True)

### D3B-22-67 · 저장할 CSV 이름 정하기

In [ ]:
forecast_csv_path = output_dir / 'kma_ultra_short_forecast.csv'

In [ ]:
forecast_csv_path

### D3B-22-68 · 예보표를 CSV로 저장하기

In [ ]:
api_forecast.to_csv(forecast_csv_path, index=False, encoding='utf-8-sig')

### D3B-22-69 · 저장된 파일 확인

In [ ]:
forecast_csv_path.exists()

### D3B-22-70 · 저장된 파일 크기 확인

In [ ]:
forecast_csv_path.stat().st_size

## 같은 결과를 웹 화면에서 보기

`app.py`는 위치 선택, 현재 예보 요약, 시간별 그래프, 표, CSV 저장 버튼을 한 화면에 배치합니다. 터미널에서 앱을 실행하면 브라우저의 로컬 주소에서 확인할 수 있습니다.

### D3B-22-71 · 웹앱 파일 위치 정하기

In [ ]:
streamlit_app_path = APP_DIR / 'app.py'

In [ ]:
streamlit_app_path

### D3B-22-72 · 웹앱 파일 확인

In [ ]:
streamlit_app_path.exists()

### D3B-22-73 · 앱에서 사용하는 기능 확인

In [ ]:
app_source = streamlit_app_path.read_text(encoding='utf-8')

In [ ]:
pd.Series({
    '위치 선택': 'selectbox' in app_source,
    '그래프': 'line_chart' in app_source,
    '표': 'dataframe' in app_source,
    'CSV 저장 버튼': 'download_button' in app_source,
    '환경변수 키': 'KMA_AUTH_KEY' in (APP_DIR / 'kma_client.py').read_text(encoding='utf-8'),
})

### D3B-22-74 · 설치 명령 확인

In [ ]:
install_command = 'python -m pip install -r apps/kma_weather_local/requirements.txt'

In [ ]:
install_command

### D3B-22-75 · 웹앱 실행 명령 확인

In [ ]:
run_command = 'python -m streamlit run apps/kma_weather_local/app.py'

In [ ]:
run_command

### D3B-22-76 · 웹앱 주소 확인

In [ ]:
local_app_url = 'http://localhost:8501'

In [ ]:
local_app_url

### D3B-22-77 · 웹앱 흐름표 만들기

In [ ]:
app_flow = pd.DataFrame({
    '순서': [1, 2, 3, 4, 5],
    '화면 동작': ['위치 선택', '응답 가져오기', '시간별 표 만들기', '그래프 표시', 'CSV 저장'],
    '사용하는 값': ['nx, ny', 'JSON', 'DataFrame', '기온·습도·풍속', '현재 표'],
})

In [ ]:
app_flow

## 웹앱에 넣을 작은 분석 만들기

습도가 높은 예보 시각을 찾는 기능을 Notebook에서 먼저 확인합니다.

### D3B-22-78 · 높은 습도 기준 정하기

In [ ]:
# 75를 70 또는 80으로 바꿔 결과 행 수를 비교하세요.
humidity_limit = 75

In [ ]:
humidity_limit

### D3B-22-79 · 높은 습도 시각 찾기

In [ ]:
humid_forecast = api_forecast[api_forecast['습도(%)'] >= humidity_limit]

In [ ]:
humid_forecast

### D3B-22-80 · 높은 습도 시각 수 확인

In [ ]:
len(humid_forecast)

### D3B-22-81 · 습도 기준선과 예보 그리기

In [ ]:
sns.lineplot(data=api_forecast, x='예보 시각', y='습도(%)', marker='o')
plt.axhline(humidity_limit, color='red', linestyle='--', label=f'{humidity_limit}%')
plt.xticks(rotation=30)
plt.legend()
plt.show()

### D3B-22-82 · 프로젝트 결과 요약 만들기

In [ ]:
project_summary = pd.Series({
    '위치': selected_location_name,
    '자료': weather_result.source,
    '예보 시작': api_forecast['예보 시각'].min(),
    '예보 끝': api_forecast['예보 시각'].max(),
    '최고 기온(°C)': api_forecast['기온(°C)'].max(),
    '높은 습도 시각 수': len(humid_forecast),
    'CSV 파일': forecast_csv_path.name,
    '웹앱 주소': local_app_url,
})

In [ ]:
project_summary

### D3B-22-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''이 Notebook의 기존 셀과 kma_client.py는 수정하지 마세요. app.py에 '습도 기준' 숫자 입력칸을 하나 추가하고, api_forecast와 같은 열 이름을 가진 forecast 표에서 기준 이상인 행 수를 화면에 표시하는 짧은 변경안을 제안해 주세요. KMA_AUTH_KEY 값을 코드·화면·오류 메시지에 표시하지 마세요. 먼저 바꿀 파일과 위치를 설명하고, 변경할 코드만 보여 주세요. 실행 뒤 사람이 확인할 항목 세 개도 적어 주세요.'''.strip()

In [ ]:
codex_request

### D3B-22-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['API 키 값이 코드나 화면에 나타나지 않는가?', '숫자 입력값을 바꾸면 높은 습도 행 수가 함께 바뀌는가?', '샘플 응답과 실제 API 응답에서 같은 열 이름을 사용하는가?', '기존 위치 선택·그래프·CSV 저장 기능이 그대로 작동하는가?'], name='확인할 내용')

In [ ]:
codex_checklist

## 프로젝트 결과물

- 기상청 응답 구조를 확인한 Notebook
- 시간별 기온·습도·풍속 그래프
- `output/kma_ultra_short_forecast.csv`
- 위치 선택과 CSV 저장이 가능한 로컬 웹앱
- 자료 출처, 선택 위치, 기준시각, 결과 해석을 담은 짧은 발표